# OpenVINO Export — Qwen3-VL 4B Receipt Model (Intel env)

Standalone companion to `finetune_qwen3vl.ipynb`, split out because the OpenVINO toolchain (**optimum-intel**) requires `transformers<5.1`, which is incompatible with **gptqmodel** (`transformers>=5.4.0`) used by the main notebook — the two can't share one Python environment.

**Run this on an Intel target (CPU / GPU / NPU).** It:
1. downloads the full-precision **bf16 master** 4B model from the Hugging Face Hub (published by the main notebook),
2. exports it to **OpenVINO IR** and int4-compresses **only the language model** (vision tower kept fp16),
3. benchmarks the int4 model on the CORD test split (same metrics as the main notebook's section 6), and
4. uploads the result to `…-receipt-extraction-openvino`.

Run it from the repo's `scripts/` dir (it reuses `utils/`), with the repo-root `hf_credentials.json` filled in.

## 0. Setup

Install the OpenVINO / optimum-intel stack (this pins `transformers<5.1`) plus the extras `utils` needs for preprocessing + metrics, and load the HF token into the environment for the private-repo download and the upload.

In [ ]:
import os
import json

# optimum-intel[openvino] pulls openvino + nncf + transformers<5.1 + torch;
# peft/datasets/zss/pillow are what utils needs to import + preprocess + score.
!pip install -q "optimum-intel[openvino]>=2.0" nncf peft datasets zss pillow

# HF token from the gitignored credentials file -> env, for the private master download + OV upload
with open("../hf_credentials.json") as f:
    _creds = json.load(f)
HF_USER = _creds["username"].strip()
HF_TOKEN = _creds["token"].strip()
os.environ["HF_TOKEN"] = HF_TOKEN
print("HF user:", HF_USER)

## 1. Load the CORD test split

For benchmarking — same preprocessing as the main notebook (`preprocess_cord`), so the metrics line up.

In [ ]:
from datasets import load_dataset
from utils import preprocess_cord, FIXED_PROMPT, aggregate_generation_metrics, DATASET_ID

dataset = load_dataset(DATASET_ID)
_, _, test_dataset = preprocess_cord(dataset, verbose=False)
print("test samples:", len(test_dataset))

## 2. Export to OpenVINO — int4 LM, fp16 vision

Download the bf16 master from the Hub and export the full pipeline to OpenVINO IR at fp16, then int4-compress **only** the `openvino_language_model` graph with NNCF so the vision tower stays fp16 (matching the GPTQ / GGUF structure).

In [ ]:
import openvino as ov
import nncf

MASTER_REPO = f"{HF_USER}/qwen3vl-4b-receipt-extraction"      # bf16 master published by the main notebook
OV_OUT = "qwen3vl-4b-receipt-extraction-openvino-int4"

# 1) download from the Hub + export the full pipeline at fp16 (optimum-cli reads HF_TOKEN for the private repo)
!optimum-cli export openvino --model {MASTER_REPO} --task image-text-to-text --weight-format fp16 {OV_OUT}

print("graphs:", [f for f in os.listdir(OV_OUT) if f.endswith(".xml")])

# 2) int4 ONLY the language-model graph; every other graph (vision, embeddings) stays fp16
lm_xml = os.path.join(OV_OUT, "openvino_language_model.xml")
lm_int4 = nncf.compress_weights(
    ov.Core().read_model(lm_xml),
    mode=nncf.CompressWeightsMode.INT4_SYM, group_size=128, ratio=1.0,
)
ov.save_model(lm_int4, lm_xml)   # overwrites the LM xml/bin; vision graphs untouched

total = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(OV_OUT) for f in fs)
print(f"OpenVINO int4 LM + fp16 VT: {total/1e9:.2f} GB in {OV_OUT}")

## 3. Benchmark on the test split

Run the int4 model over the test split and report the same metrics as the main notebook's section 6 (`field_f1`, `normalized_ted`, `json_validity`, `exact_match`), so it's directly comparable to the 4B GPTQ (field_f1 0.8921) and the GGUF Q4_K_M.

Set `OV_DEVICE` to `"CPU"`, `"GPU"`, or `"NPU"` for your Intel target.

In [ ]:
from optimum.intel import OVModelForVisualCausalLM
from transformers import AutoProcessor
from tqdm.auto import tqdm

OV_DEVICE = "CPU"   # "GPU" / "NPU" for Intel accelerators
MAX_PX = 1600       # match the training/export resolution

ov_model = OVModelForVisualCausalLM.from_pretrained(OV_OUT, device=OV_DEVICE)
ov_processor = AutoProcessor.from_pretrained(OV_OUT, max_pixels=MAX_PX * 28 * 28)

ov_preds = []
for ex in tqdm(test_dataset, desc="ov-eval", unit="img"):
    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": FIXED_PROMPT}]}]
    text = ov_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = ov_processor(text=[text], images=[ex["image"]], return_tensors="pt")
    out = ov_model.generate(**inputs, max_new_tokens=512, do_sample=False)
    gen = out[:, inputs["input_ids"].shape[1]:]
    ov_preds.append(ov_processor.batch_decode(gen, skip_special_tokens=True)[0].strip())

ov_metrics = aggregate_generation_metrics(ov_preds, [ex["label"] for ex in test_dataset])
print(f"[OpenVINO int4]  field_f1={ov_metrics['field_f1']:.4f}  "
      f"normalized_ted={ov_metrics['normalized_ted']:.4f}  "
      f"json_validity={ov_metrics['json_validity']:.4f}  "
      f"exact_match={ov_metrics['exact_match']:.4f}")

## 4. Publish to the Hugging Face Hub

Upload the OpenVINO int4 model to a private `…-receipt-extraction-openvino` repo.

In [ ]:
from huggingface_hub import HfApi

repo_id = f"{HF_USER}/qwen3vl-4b-receipt-extraction-openvino"
api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id, private=True, exist_ok=True)
api.upload_folder(folder_path=OV_OUT, repo_id=repo_id, commit_message="OpenVINO int4 (LM) + fp16 vision")
print(f"Uploaded {OV_OUT} -> https://huggingface.co/{repo_id} (private)")